# `WaveFunction` — Classe mère abstraite

**Fichier source :** `src/quantum_sim/waves/wave_function.py`

---

## 1. Contexte physique : qu'est-ce qu'une fonction d'onde ?

En mécanique quantique, l'état d'une particule est entièrement décrit par sa **fonction d'onde** $\psi(x, t)$, une fonction complexe de la position $x$ et du temps $t$.

Elle n'est pas directement observable, mais son module au carré donne la **densité de probabilité de présence** :

$$
\rho(x, t) = |\psi(x, t)|^2
$$

Cette densité de probabilité doit satisfaire la **condition de normalisation** :

$$
\int_{-\infty}^{+\infty} |\psi(x, t)|^2 \, dx = 1
$$

ce qui signifie que la probabilité de trouver la particule quelque part dans l'espace est de 100 %.

---

## 2. Rôle de la classe `WaveFunction`

`WaveFunction` est une **classe abstraite** (au sens Python `ABC`) qui définit le **contrat commun** à tous les types d'ondes du projet. Elle impose :

| Méthode / attribut | Rôle |
|---|---|
| `position` | Position spatiale de référence $x_0$ |
| `time` | Paramètre temporel $t$ |
| `validate_parameters()` | Validation physique des paramètres (abstraite) |
| `evaluate(x)` | Calcul de $\psi(x, t)$ (abstraite) |
| `evaluate_interval(start, end, points)` | Évaluation sur une grille régulière |
| `probability_density(x)` | Calcul de $|\psi(x)|^2$ |

---

## 3. Méthodes concrètes

### 3.1 `evaluate_interval`

Génère un tableau de positions régulièrement espacées avec `np.linspace`, puis appelle `evaluate`. Cela correspond à un **échantillonnage spatial** :

$$
x_i = x_{\text{start}} + i \cdot \frac{x_{\text{end}} - x_{\text{start}}}{N - 1}, \quad i = 0, \ldots, N-1
$$

### 3.2 `probability_density`

Calcule $|\psi(x)|^2 = \psi(x) \cdot \psi^*(x)$ via `np.abs(psi)**2`. En Python :
- `psi` est un tableau `complex128` (partie réelle + imaginaire)
- `np.abs` calcule le module : $|a + ib| = \sqrt{a^2 + b^2}$

---

## 4. Code source annoté

In [2]:
from abc import ABC, abstractmethod
import numpy as np

class WaveFunction(ABC):
    """Classe de base abstraite pour les fonctions d'onde quantiques."""

    def __init__(self, position: float | None, time: float = 0.0):
        """
        position : position spatiale de référence x₀
        time     : paramètre temporel t
        """
        self.position = position
        self.time = time
        self.validate_parameters()  # Validation immédiate à la construction

    @abstractmethod
    def validate_parameters(self) -> None:
        """Chaque sous-classe doit valider ses propres paramètres."""
        pass

    @abstractmethod
    def evaluate(self, x: float | np.ndarray) -> np.ndarray:
        """
        Calcule ψ(x, t) pour une ou plusieurs positions.
        Retourne un tableau complexe.
        """
        pass

    def evaluate_interval(self, start: float, end: float, points: int):
        """
        Évalue ψ sur une grille uniforme de `points` points entre `start` et `end`.
        Retourne (x, ψ(x)).
        """
        x = np.linspace(start, end, points)
        return x, self.evaluate(x)

    def probability_density(self, x: float | np.ndarray) -> np.ndarray:
        """
        Calcule la densité de probabilité |ψ(x)|².
        """
        psi = self.evaluate(x)
        return np.abs(psi) ** 2

## 5. Le patron de conception : Méthode Template

`WaveFunction` applique le **patron Template Method** :
- `evaluate_interval` et `probability_density` sont des **algorithmes fixes** définis dans la base
- Ils délèguent le calcul physique à `evaluate`, qui est **abstraite**

Cela garantit que toutes les sous-classes (`PlaneWave`, `WavePacket`, `GaussianWavePacket`) partagent le même comportement pour les opérations communes, sans dupliquer de code.

```
WaveFunction (ABC)
│
├── evaluate()            ← à implémenter
├── validate_parameters() ← à implémenter
├── evaluate_interval()   ← utilise evaluate()
└── probability_density() ← utilise evaluate()
```